# **Build the fist MIL pipeline : 1D CNN + Classifier**

## **Defining the training on the GPU**

In [1]:
from pathlib import Path
CHECKPOINT_DIR = Path("checkpoints_mil_classifier_3s")


In [2]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Dossier des checkpoints :", CHECKPOINT_DIR.resolve())

Dossier des checkpoints : C:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\Base_line\checkpoints_mil_classifier_3s


In [3]:
import torch.nn as nn
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [4]:
import os
# Enable precise CUDA kernel error reporting
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# Import PyTorch AFTER setting the environment variables
import torch

In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing to train on CPU.")
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## **Building the dataset**

In [6]:
from Data_pre_processing.data_cleaning import data_cleaning, split_train_test
import torch
from torch.utils.data import Dataset
from Data_pre_processing.Signal_treatement import split_signals_and_build

### **Getting the data**

La fonction upload_data retourn deux tuples : 
- Le premier tuple (training_x, training_y)
- Le deuxieme tuple (testing_x, testing_y)

donc pour acceder il suffit d'utiliser les indices 0 et 1, exemple : 

- training[0] donne le training_x, training[1] donne le training_y

In [7]:
from Data_pre_processing.Data_pipeline import upload_data

In [8]:
training, testing = upload_data(segment_duration=3) 

Normal: 522
Abnormal: 426


### **Building the dataset using PyTorch for training**

In [9]:
from Data_pre_processing.Build_dataset import PCGDataset

In [10]:
Training_data = PCGDataset(bags = training[0], labels = training[1])
Testing_data = PCGDataset(bags = testing[0], labels = testing[1])

## **Training**

- Pour l'entrainement on a le problème d'umblanced data, donc pour valider le modele, on vas utiliser le stratified cross validation.
- Le nombre de folds qu'on vas utiliser maintenant c'est 5.
- 1 pour abnormal, 0 pour normal.
- In this case we are going to priviluaze our folds to see the data distribution

### **Stratified cross validation**

In [11]:
skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

### **Training phase**

In [12]:
from Data_pre_processing.Build_dataset import build_data_loaders
from Models.MIL1D import Mil_classifier
from Training.Train_model import TrainModel

#### **Training and validation**

Ici je défini les Hyper parametres :

In [ ]:
NUM_EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
fold_results = []

Phase d'entrainement :

In [14]:
folds = []
pos_weight = torch.tensor([0.8], device=device)

for train_index, test_index in skf.split(
    np.zeros(len(training[1])),
    training[1],
):
    folds.append((train_index, test_index))

for fold, (train_idx, valid_idx) in enumerate(folds) : 
    print(f"training on the fold {fold}")
    print(f"*"*20, "INFORMATION ABOUT THE TRAINING", "*"*20)

    numbers_of_zero = 0
    numbers_of_one = 0
    for index in train_index :
      if training[1][index] == 1 :
        numbers_of_one+=1
      else : numbers_of_zero+=1
    print(f"      numbers of abnormal : {numbers_of_one}, numbers of normal : {numbers_of_zero}")

    torch.manual_seed(100 + fold)

    model = Mil_classifier().to(device=device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    checkpoint_path = CHECKPOINT_DIR / f"fold_{fold}.pt"

    start_epoch = 0

    if checkpoint_path.exists():
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )

        if checkpoint["fold"] != fold:
            raise ValueError("Le checkpoint ne correspond pas au fold.")

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        print(
            f"Reprise du fold {fold} : "
            f"dernière époque terminée = {start_epoch}"
        )
    else:
        print(f"New training for the fold {fold}")

    train_loader, valid_loader = build_data_loaders(dataset=Training_data, train_index=train_index, test_index=test_index)
    
    Training_model = TrainModel(
       model=model,
       criterion=criterion,
       optimizer=optimizer,
       train_loader=train_loader,
       device=device,
       valid_loader=valid_loader,
       start_epoch=0,
       number_epochs=NUM_EPOCHS,
       checkpoint_path=checkpoint_path
    )

    Training_model.train_model(fold=fold)
    

training on the fold 0
******************** INFORMATION ABOUT THE TRAINING ********************
      numbers of abnormal : 284, numbers of normal : 348
New training for the fold 0
checkpoint saved .. !
Fold : 0 | Train loss : 0.6259 | Validation loss : 0.6074 | epoch : 1/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.6057 | Validation loss : 0.5816 | epoch : 2/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.6003 | Validation loss : 0.5745 | epoch : 3/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.5993 | Validation loss : 0.5724 | epoch : 4/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.5983 | Validation loss : 0.5736 | epoch : 5/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.5967 | Validation loss : 0.5739 | epoch : 6/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.5951 | Validation loss : 0.5736 | epoch : 7/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.5934 | Validation loss : 0.5722 | epoch : 8/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.5915 | Vali

KeyboardInterrupt: 